## Predictions of COCO-trained model

This stage continues the preparation for evaluation by generating predictions using the COCO-trained EoMT model on the same Cityscapes validation dataset.

The COCO-trained model is optimized for panoptic segmentation. Therefore, its outputs contain both semantic category information and instance-level segmentation. In order to enable a fair semantic segmentation comparison, the panoptic predictions must later be converted into a semantic representation compatible with the Cityscapes label space.

As in the previous stage (predictions_eomt_city), predictions are saved in .png format which is a standard semantic segmentation output format convenient for fair evaluation.

*Inference code was taken and adapted from eomt/inference.ipynb*

In [1]:
import yaml
from lightning import seed_everything
import torch
import numpy as np
import warnings
from torch.nn import functional as F
from torch.amp.autocast_mode import autocast
import matplotlib.pyplot as plt
import importlib
from pathlib import Path
from tqdm import tqdm
import os
from PIL import Image


warnings.filterwarnings("ignore")

seed_everything(0, verbose=False)

device = 0

/home/elisa/outlierdrive/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Notice

In order to load the COCO-trained model, use COCO configuration file. However for loading Cityscapes validation dataset, keep Cityscapes configuration file. Because we need to run COCO-trained model on Cityscapes validation set.

In [2]:
coco_config_path = "../../eomt/configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml"
city_config_path = "../../eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"

trained_city_path = "../../eomt_checkpoints/eomt_cityscapes.bin"
trained_coco_path = "../../eomt_checkpoints/eomt_coco.bin"

data_path = "../../data/datasets"

output_coco_path_npy = "../../data/eomt_valset_predictions/coco_model/npy"
output_coco_path_png = "../../data/eomt_valset_predictions/coco_model/png"

os.makedirs(output_coco_path_npy, exist_ok=True)
os.makedirs(output_coco_path_png, exist_ok=True)


with open(city_config_path, "r") as f:
    city_config = yaml.safe_load(f)

with open(coco_config_path, "r") as f:
    coco_config = yaml.safe_load(f)

---

## Load the data

In [ ]:
import sys
sys.path.append("../../eomt")

In [5]:
print(city_config["data"]["class_path"])

datasets.cityscapes_semantic.CityscapesSemantic


In [8]:
def load_data(config, data_path):
    data_module_name, class_name = config["data"]["class_path"].rsplit(".", 1)
    data_module = getattr(importlib.import_module(data_module_name), class_name)
    data_module_kwargs = config["data"].get("init_args", {})

    return data_module(
        path=data_path,
        batch_size=1,
        num_workers=0,
        check_empty_targets=False,
        **data_module_kwargs
    ).setup()

# download only Cityscapes validation images
# COCO model is used only for inference now
city_data = load_data(city_config, data_path)

Since we already deeply explored the dataset in predictions_eomt_city.ipynb, it should not be repeated again. We only conduct small check to make sure that dataset is loaded correctly.

In [9]:
val_dataset = city_data.val_dataloader().dataset

print("Validation set size:", len(val_dataset))
print(val_dataset.imgs[0])

Validation set size: 500
leftImg8bit/val/frankfurt/frankfurt_000000_000294_leftImg8bit.png


---

## Load the model

In [ ]:
def build_model(config, device, data=None, reference_img_size=None, reference_num_classes=None):

    warnings.filterwarnings(
        "ignore",
        message=r".*Attribute 'network' is an instance of `nn\.Module` and is already saved during checkpointing.*",
    )

    # data=None for COCO model without loading COCO dataset

    if data is not None:
        img_size = data.img_size
        num_classes = data.num_classes
    else:
        img_size = reference_img_size
        num_classes = reference_num_classes

    # Load encoder
    encoder_cfg = config["model"]["init_args"]["network"]["init_args"]["encoder"]
    encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
    encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)

    encoder = encoder_cls(
        img_size=img_size,
        **encoder_cfg.get("init_args", {})
    )

    # Load network
    network_cfg = config["model"]["init_args"]["network"]
    network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
    network_cls = getattr(importlib.import_module(network_module_name), network_class_name)

    network_kwargs = {
        k: v for k, v in network_cfg["init_args"].items() if k != "encoder"
    }

    network = network_cls(
        masked_attn_enabled=False,
        num_classes=num_classes,
        encoder=encoder,
        **network_kwargs,
    )

    # Load Lightning module
    lit_module_name, lit_class_name = config["model"]["class_path"].rsplit(".", 1)
    lit_cls = getattr(importlib.import_module(lit_module_name), lit_class_name)

    model_kwargs = {
        k: v for k, v in config["model"]["init_args"].items() if k != "network"
    }

    if "stuff_classes" in config["data"].get("init_args", {}):
        model_kwargs["stuff_classes"] = config["data"]["init_args"]["stuff_classes"]

    model = (
        lit_cls(
            img_size=img_size,
            num_classes=num_classes,
            network=network,
            **model_kwargs,
        )
        .eval()
        .to(device)
    )

    return model

In [11]:
coco_model = build_model(
    coco_config,
    device,
    data=None,
    reference_img_size=(640, 640),
    reference_num_classes=133
)

## Load pre-trained weights

In [12]:
def load_weights(model, checkpoint, device):
    weights = torch.load(
        checkpoint,
        map_location=f"cuda:{device}",
        weights_only=False
    )

    if "state_dict" in weights:
        weights = weights["state_dict"]

    model.load_state_dict(weights, strict=False)

    return model

In [13]:
coco_model_trained = load_weights(coco_model, trained_coco_path, device)

In [14]:
print(type(coco_model_trained))
print(coco_model_trained)

<class 'training.mask_classification_panoptic.MaskClassificationPanoptic'>
MaskClassificationPanoptic(
  (network): EoMT(
    (encoder): ViT(
      (backbone): VisionTransformer(
        (patch_embed): PatchEmbed(
          (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
          (norm): Identity()
        )
        (pos_drop): Dropout(p=0.0, inplace=False)
        (patch_drop): Identity()
        (norm_pre): Identity()
        (blocks): Sequential(
          (0): Block(
            (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
            (attn): Attention(
              (qkv): Linear(in_features=768, out_features=2304, bias=True)
              (q_norm): Identity()
              (k_norm): Identity()
              (attn_drop): Dropout(p=0.0, inplace=False)
              (norm): Identity()
              (proj): Linear(in_features=768, out_features=768, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
            )
           

--- 

## Panoptic inference

In [15]:
def infer_panoptic(img, model):
    
    with torch.inference_mode(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]

        transformed_imgs = model.resize_and_pad_imgs_instance_panoptic(imgs)
        mask_logits_per_layer, class_logits_per_layer = model(transformed_imgs)

        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], model.img_size, mode="bilinear"
        )

        mask_logits = model.revert_resize_and_pad_logits_instance_panoptic(
            mask_logits, img_sizes
        )

        preds = model.to_per_pixel_preds_panoptic(
            mask_logits,
            class_logits_per_layer[-1],
            model.stuff_classes,
            model.mask_thresh,
            model.overlap_thresh,
        )[0].cpu()

    pred = preds.numpy()
    sem_pred, inst_pred = pred[..., 0], pred[..., 1]

    return sem_pred, inst_pred

We conduct the check on one validation sample. Panoptic inference generates semantic prediction: semantic category id per pixel and instance prediction: instance id per pixel.

Also make sure that semantic prediction is a label map. It will be important for later evaluation.

Notice: we can see that model predicts COCO panoptic category ids and Cityscapes trainIds. So we have to conduct class-space mapping.

In [16]:
val_dataset = city_data.val_dataloader().dataset

img, target = val_dataset[0]

sem_pred_coco, inst_pred = infer_panoptic(img, coco_model_trained)

print(sem_pred_coco.shape, sem_pred_coco.dtype)
print(inst_pred.shape, inst_pred.dtype)
print("COCO semantic ids:", np.unique(sem_pred_coco))
print("Instance ids:", np.unique(inst_pred)[:20])

(1024, 2048) int64
(1024, 2048) int64
COCO semantic ids: [  0   2 100 116 119 123 129 133]
Instance ids: [-1  0  1  2  3  4  5  6  7  8  9]


Retrieve explicit Cityscapes label names. We can understand which Cityscapes classes are valid for evaluation, which trainIds correpsond to each semantic category and which labels should be ignored.

TrainIds are used during traing and evaluation, so we will work with them for fair evaluation. As we can notice, only 0-18 trainIds are valid evaluation classes, so everything else should be ignored.

For the methodology, this step carefully inspects official Cityscapes label definitions, trainIds and ignored labels.

In [17]:
from torchvision.datasets import Cityscapes

for cls in Cityscapes.classes:
    print(cls.name, "id:", cls.id, "train_id:", cls.train_id, "ignore:", cls.ignore_in_eval)

unlabeled id: 0 train_id: 255 ignore: True
ego vehicle id: 1 train_id: 255 ignore: True
rectification border id: 2 train_id: 255 ignore: True
out of roi id: 3 train_id: 255 ignore: True
static id: 4 train_id: 255 ignore: True
dynamic id: 5 train_id: 255 ignore: True
ground id: 6 train_id: 255 ignore: True
road id: 7 train_id: 0 ignore: False
sidewalk id: 8 train_id: 1 ignore: False
parking id: 9 train_id: 255 ignore: True
rail track id: 10 train_id: 255 ignore: True
building id: 11 train_id: 2 ignore: False
wall id: 12 train_id: 3 ignore: False
fence id: 13 train_id: 4 ignore: False
guard rail id: 14 train_id: 255 ignore: True
bridge id: 15 train_id: 255 ignore: True
tunnel id: 16 train_id: 255 ignore: True
pole id: 17 train_id: 5 ignore: False
polegroup id: 18 train_id: 255 ignore: True
traffic light id: 19 train_id: 6 ignore: False
traffic sign id: 20 train_id: 7 ignore: False
vegetation id: 21 train_id: 8 ignore: False
terrain id: 22 train_id: 9 ignore: False
sky id: 23 train_id: 10

Check which semantic categories the COCO-trained panoptic model actually predicts on a Cityscapes validation image. Moreover, it confirms that the model is producing valid semantic category predictions.

In [18]:
print(np.unique(sem_pred_coco))

[  0   2 100 116 119 123 129 133]


At this moment the direct evaluation against Cityscapes ground truth is impossible without class-space alignment.

---

## COCO -> Cityscapes mapping

Map predictions from COCO ids to Cityscapes trainIds and save only the mapped semantic masks. Verify the classes overlap manually.

Firstly, we check the file eomt/datasets/coc_panoptic.py which contains the mapping from original COCO category ids to the model’s internal contiguous ids. So sem_pred_coco contains the values after this mapping, meaning internal ids.

Then we list the trainIds of Cityscapes dataset

In [19]:
from torchvision.datasets import Cityscapes

for cls in Cityscapes.classes:
    
    if cls.train_id != 255 and cls.train_id != -1:
        
        print(
            f"name={cls.name:15} "
            f"| train_id={cls.train_id}"
        )

name=road            | train_id=0
name=sidewalk        | train_id=1
name=building        | train_id=2
name=wall            | train_id=3
name=fence           | train_id=4
name=pole            | train_id=5
name=traffic light   | train_id=6
name=traffic sign    | train_id=7
name=vegetation      | train_id=8
name=terrain         | train_id=9
name=sky             | train_id=10
name=person          | train_id=11
name=rider           | train_id=12
name=car             | train_id=13
name=truck           | train_id=14
name=bus             | train_id=15
name=train           | train_id=16
name=motorcycle      | train_id=17
name=bicycle         | train_id=18


Now we manually explore the official COCO Panoptic category definitions provided in the 2017 annotation files. The final evaluation is performed on the Cityscapes validation set, so we do not need to consider all COCO categories. 

We only include manually selected classes that have a meaningful semantic correspondence with Cityscapes classes.

In [ ]:

COCO_CATEGORIES = {
    1: "person",
    2: "bicycle",
    3: "car",
    4: "motorcycle",
    6: "bus",
    7: "train",
    8: "truck",
    10: "traffic light",
    13: "stop sign",
    149: "road",
    191: "pavement-merged",
    197: "building-other-merged",

    171: "wall-brick",
    175: "wall-stone",
    176: "wall-tile",
    177: "wall-wood",
    199: "wall-other-merged",

    185: "fence-merged",

    184: "tree-merged",
    193: "grass-merged",

    187: "sky-other-merged",
}


## Notice 
In order to map COCO classes to Cityscapes trainIds, we need to retrieve **internal EoNT id** based on original COCO id. We inspected it manually in coco_panoptic.py.

*If necessary, add your absolute path**

In [ ]:

import sys
sys.path.append("/home/elisa/outlierdrive")

from eomt.datasets.coco_panoptic import CLASS_MAPPING

for coco_id, class_name in COCO_CATEGORIES.items():

    internal_id = CLASS_MAPPING[coco_id]

    print(
        f"class={class_name:<15} "
        f"| original COCO id={coco_id:<3} "
        f"| internal EoMT id={internal_id}"
    )

class=person          | original COCO id=1   | internal EoMT id=0
class=bicycle         | original COCO id=2   | internal EoMT id=1
class=car             | original COCO id=3   | internal EoMT id=2
class=motorcycle      | original COCO id=4   | internal EoMT id=3
class=bus             | original COCO id=6   | internal EoMT id=5
class=train           | original COCO id=7   | internal EoMT id=6
class=truck           | original COCO id=8   | internal EoMT id=7
class=traffic light   | original COCO id=10  | internal EoMT id=9
class=stop sign       | original COCO id=13  | internal EoMT id=11
class=road            | original COCO id=149 | internal EoMT id=100
class=pavement-merged | original COCO id=191 | internal EoMT id=123
class=building-other-merged | original COCO id=197 | internal EoMT id=129
class=wall-brick      | original COCO id=171 | internal EoMT id=109
class=wall-stone      | original COCO id=175 | internal EoMT id=110
class=wall-tile       | original COCO id=176 | internal EoM

In [20]:
from eomt.datasets.coco_panoptic import CLASS_MAPPING

print("Raw COCO ids used by EoMT:")
print(sorted(CLASS_MAPPING.keys()))

Raw COCO ids used by EoMT:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 27, 28, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 67, 70, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 84, 85, 86, 87, 88, 89, 90, 92, 93, 95, 100, 107, 109, 112, 118, 119, 122, 125, 128, 130, 133, 138, 141, 144, 145, 147, 148, 149, 151, 154, 155, 156, 159, 161, 166, 168, 171, 175, 176, 177, 178, 180, 181, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200]


Now we create a map over overlapped class space. This is the most crucial step for evaluation since later we need to run the COCO-trained model over all validation set and save the predictions over mapped class space.

In [ ]:
IGNORE_INDEX = 255

COCO_TO_CITYSCAPES_TRAINID = {
    0: 11,   # person -> person
    1: 18,   # bicycle -> bicycle
    2: 13,   # car -> car
    3: 17,   # motorcycle -> motorcycle
    5: 15,   # bus -> bus
    6: 16,   # train -> train
    7: 14,   # truck -> truck
    9: 6,    # traffic light -> traffic light
    11: 7,   # # stop sign -> traffic sign (approximate semantic overlap)

     
    100: 0,   # road
    123: 1,   # pavement-merged -> sidewalk
    129: 2,   # building-other-merged -> building

    109: 3,   # wall-brick
    110: 3,   # wall-stone
    111: 3,   # wall-tile
    112: 3,   # wall-wood
    131: 3,   # wall-other-merged

    117: 4,   # fence-merged

    116: 8,   # tree-merged
    125: 8,   # grass-merged

    119: 10,  # sky-other-merged

}

def map_coco_to_cityscapes(sem_pred_coco):
    mapped = np.full(
        sem_pred_coco.shape,
        IGNORE_INDEX,
        dtype=np.uint8
    )

    for coco_id, city_train_id in COCO_TO_CITYSCAPES_TRAINID.items():
        mapped[sem_pred_coco == coco_id] = city_train_id

    return mapped

In [22]:
img, target = val_dataset[0]

sem_pred_coco, inst_pred = infer_panoptic(img, coco_model_trained)

mapped_pred = map_coco_to_cityscapes(sem_pred_coco)

print("Raw COCO ids:", np.unique(sem_pred_coco))
print("Mapped Cityscapes trainIds:", np.unique(mapped_pred))
print(mapped_pred.shape, mapped_pred.dtype)

Raw COCO ids: [  0   2 100 116 119 123 129 133]
Mapped Cityscapes trainIds: [  0   1   2   8  10  11  13 255]
(1024, 2048) uint8


Run the following code in Colab to save the predictions .png:

In [ ]:
print("Validation set size:", len(val_dataset))

for idx in tqdm(range(len(val_dataset))):

    img, target = val_dataset[idx]

    sem_pred_coco, inst_pred = infer_panoptic(
        img,
        coco_model_trained
    )

    mapped_pred = map_coco_to_cityscapes(
        sem_pred_coco
    )

    original_path = val_dataset.imgs[idx]
    original_name = os.path.basename(original_path)

    base_name = original_name.replace(
        "_leftImg8bit.png",
        ""
    )

    png_name = f"{base_name}_predTrainIds.png"

    Image.fromarray(mapped_pred).save(
        os.path.join(
            output_coco_path_png,
            png_name
        )
    )

print("Finished saving mapped COCO predictions")

---

## Predicted classes

In [ ]:
from glob import glob
import os

coco_pngs = sorted(
    glob("../../data/eomt_valset_predictions/coco_model/png/content/drive/MyDrive/eomt_valset_predictions/png/*.png")
)

print("Number of saved predictions:", len(coco_pngs))

Number of saved predictions: 500


In [26]:
all_mapped_values = set()

for path in coco_pngs:
    mask = np.array(Image.open(path))
    all_mapped_values.update(np.unique(mask).tolist())

print(sorted(all_mapped_values))

[0, 1, 2, 3, 4, 6, 7, 8, 10, 11, 13, 14, 15, 17, 18, 255]


--- 
## Summary

At this stage we performed inference using the COCO Panoptic-trained EoMT model on the Cityscapes validation set and prepared its predictions for a fair semantic segmentation comparison with the Cityscapes-trained model. 

To better understand the behavior of the COCO-trained model, we also analyzed its prediction behaviour.

A detailed inspection showed that many classes were classified fully, which highlights the effect of domain shift.

In the following stage we will build the consistent evaluation pipeline and compare both models on a common semantic label space.